## **Aim**
To implement a program that analyzes a simulated malware activity log and identifies suspicious file, process, and network behavior.

## **Algorithm**
**Step 1:** Import `json`, `collections.Counter`, `datetime`, and `re` libraries.

**Step 2:** Create a simulated malware activity log (JSON) with events for:
   - File operations (create, modify, delete, encrypt)
   - Process execution (spawn, inject, hollow)
   - Network connections (C2, download, exfil)
   - Registry modifications (persistence, config)
   - Memory operations (injection, dumping)

**Step 3:** Parse events and categorize by type.

**Step 4:** Define behavioral indicators for malware:
   - Ransomware: mass file encryption, ransom notes
   - Spyware: keylogging, screen capture, data collection
   - Trojan: C2 communication, payload download
   - Worm: lateral movement, self-propagation
   - Rootkit: hidden files, process hiding, hooking
   - Miner: high CPU, GPU usage, pool connections

**Step 5:** Score each process/entity based on indicators.

**Step 6:** Generate malware behavior report with classification.

In [1]:
import json
from collections import Counter, defaultdict
from datetime import datetime, timedelta

def create_sample_malware_log(log_file):
    now = datetime.now()
    base = now - timedelta(hours=2)
    
    events = []
    
    # Ransomware activity
    for i in range(20):
        events.append({
            "timestamp": (base + timedelta(minutes=i*2)).isoformat(),
            "type": "FILE",
            "process": "malware.exe",
            "pid": 5678,
            "operation": "ENCRYPT",
            "file": f"C:\\Users\\user\\Documents\\doc_{i}.docx",
            "new_extension": ".encrypted",
            "entropy": 7.9
        })
    
    events.append({
        "timestamp": (base + timedelta(minutes=45)).isoformat(),
        "type": "FILE",
        "process": "malware.exe",
        "pid": 5678,
        "operation": "CREATE",
        "file": "C:\\Users\\user\\README_DECRYPT.txt",
        "content": "YOUR FILES ARE ENCRYPTED. PAY 1 BTC."
    })
    
    events.append({
        "timestamp": (base + timedelta(minutes=50)).isoformat(),
        "type": "PROCESS",
        "process": "malware.exe",
        "pid": 5678,
        "operation": "SHADOW_COPY_DELETE",
        "command": "vssadmin delete shadows /all /quiet"
    })
    
    events.append({
        "timestamp": (base + timedelta(minutes=55)).isoformat(),
        "type": "REGISTRY",
        "process": "malware.exe",
        "pid": 5678,
        "operation": "SET_VALUE",
        "key": "HKCU\\Software\\Microsoft\\Windows\\CurrentVersion\\Run",
        "value": "WindowsUpdate",
        "data": "C:\\Users\\user\\AppData\\Roaming\\malware.exe"
    })
    
    for i in range(5):
        events.append({
            "timestamp": (base + timedelta(minutes=60+i*5)).isoformat(),
            "type": "NETWORK",
            "process": "malware.exe",
            "pid": 5678,
            "operation": "CONNECT",
            "dest_ip": "192.168.100.50",
            "dest_port": 443,
            "protocol": "HTTPS",
            "bytes_sent": 1000,
            "bytes_received": 500
        })
    
    # Keylogger activity
    events.append({
        "timestamp": (base + timedelta(minutes=10)).isoformat(),
        "type": "PROCESS",
        "process": "keylogger.exe",
        "pid": 4321,
        "operation": "KEYBOARD_HOOK",
        "api": "SetWindowsHookEx"
    })
    
    events.append({
        "timestamp": (base + timedelta(minutes=15)).isoformat(),
        "type": "PROCESS",
        "process": "keylogger.exe",
        "pid": 4321,
        "operation": "SCREEN_CAPTURE",
        "api": "BitBlt"
    })
    
    for i in range(5):
        events.append({
            "timestamp": (base + timedelta(minutes=20+i*5)).isoformat(),
            "type": "FILE",
            "process": "keylogger.exe",
            "pid": 4321,
            "operation": "WRITE",
            "file": f"C:\\Temp\\staging\\keylog_{i}.dat",
            "size": 10240
        })
    
    events.append({
        "timestamp": (base + timedelta(minutes=50)).isoformat(),
        "type": "REGISTRY",
        "process": "keylogger.exe",
        "pid": 4321,
        "operation": "SCHEDULED_TASK",
        "task": "WindowsDefenderUpdate",
        "command": "C:\\Temp\\keylogger.exe"
    })
    
    for i in range(3):
        events.append({
            "timestamp": (base + timedelta(minutes=55+i*5)).isoformat(),
            "type": "NETWORK",
            "process": "keylogger.exe",
            "pid": 4321,
            "operation": "EXFILTRATE",
            "dest_ip": "203.0.113.45",
            "dest_port": 8080,
            "protocol": "HTTP",
            "bytes_sent": 50000,
            "bytes_received": 100
        })
    
    # Trojan downloader
    events.append({
        "timestamp": (base + timedelta(minutes=5)).isoformat(),
        "type": "NETWORK",
        "process": "trojan_downloader.exe",
        "pid": 7890,
        "operation": "DOWNLOAD",
        "url": "http://192.168.100.50/payload.bin",
        "dest_ip": "192.168.100.50",
        "size": 500000
    })
    
    events.append({
        "timestamp": (base + timedelta(minutes=10)).isoformat(),
        "type": "PROCESS",
        "process": "trojan_downloader.exe",
        "pid": 7890,
        "operation": "PROCESS_INJECTION",
        "target": "explorer.exe",
        "api": "CreateRemoteThread"
    })
    
    events.append({
        "timestamp": (base + timedelta(minutes=15)).isoformat(),
        "type": "REGISTRY",
        "process": "trojan_downloader.exe",
        "pid": 7890,
        "operation": "DISABLE_DEFENDER",
        "key": "HKLM\\SOFTWARE\\Microsoft\\Windows Defender",
        "value": "DisableAntiSpyware",
        "data": 1
    })
    
    events.append({
        "timestamp": (base + timedelta(minutes=20)).isoformat(),
        "type": "REGISTRY",
        "process": "trojan_downloader.exe",
        "pid": 7890,
        "operation": "FIREWALL_RULE",
        "rule": "AllowMalwareOutbound",
        "action": "Allow",
        "port": 443
    })
    
    events.append({
        "timestamp": (base + timedelta(minutes=25)).isoformat(),
        "type": "REGISTRY",
        "process": "trojan_downloader.exe",
        "pid": 7890,
        "operation": "SERVICE_CREATE",
        "service": "WindowsUpdateService",
        "binary": "C:\\Windows\\Temp\\payload.exe"
    })
    
    # Cryptominer
    for i in range(5):
        events.append({
            "timestamp": (base + timedelta(minutes=30+i*10)).isoformat(),
            "type": "NETWORK",
            "process": "cryptominer.exe",
            "pid": 9012,
            "operation": "MINING_POOL",
            "dest_ip": "192.168.100.100",
            "dest_port": 3333,
            "protocol": "STRATUM",
            "bytes_sent": 1000,
            "bytes_received": 2000
        })
    
    events.append({
        "timestamp": (base + timedelta(minutes=35)).isoformat(),
        "type": "PROCESS",
        "process": "cryptominer.exe",
        "pid": 9012,
        "operation": "HIGH_CPU",
        "cpu_percent": 95
    })
    
    # Suspicious update
    events.append({
        "timestamp": (base + timedelta(minutes=5)).isoformat(),
        "type": "FILE",
        "process": "suspicious_update.exe",
        "pid": 1023,
        "operation": "EXECUTE",
        "file": "C:\\Temp\\suspicious_update.exe",
        "signed": False
    })
    
    with open(log_file, "w") as f:
        json.dump(events, f, indent=2)

def analyze_malware_log(log_file):
    with open(log_file, "r") as f:
        events = json.load(f)
    
    # Group by process
    process_events = defaultdict(list)
    for e in events:
        process_events[e["process"]].append(e)
    
    process_scores = {}
    
    for proc, procs_events in process_events.items():
        score = 0
        indicators = []
        pid = procs_events[0].get("pid", 0)
        
        # File operations
        file_ops = [e for e in procs_events if e["type"] == "FILE"]
        encrypt_ops = [e for e in file_ops if e.get("operation") == "ENCRYPT"]
        if len(encrypt_ops) >= 10:
            score += 40
            indicators.append(f"Mass file encryption ({len(encrypt_ops)} files)")
        
        ransom_notes = [e for e in file_ops if "README_DECRYPT" in e.get("file", "") or "ransom" in e.get("content", "").lower()]
        if ransom_notes:
            score += 30
            indicators.append(f"Ransom note dropped ({len(ransom_notes)})")
        
        high_entropy = [e for e in file_ops if e.get("entropy", 0) > 7.5]
        if high_entropy:
            score += 15
            indicators.append(f"High entropy file writes ({len(high_entropy)})")
        
        # Process operations
        proc_ops = [e for e in procs_events if e["type"] == "PROCESS"]
        injections = [e for e in proc_ops if e.get("operation") == "PROCESS_INJECTION"]
        if injections:
            score += 20
            indicators.append(f"Process injection ({len(injections)})")
        
        hooks = [e for e in proc_ops if e.get("operation") in ["KEYBOARD_HOOK", "SCREEN_CAPTURE"]]
        if hooks:
            score += 20
            indicators.append(f"Keyboard/screen hooks ({len(hooks)})")
        
        cpu_ops = [e for e in proc_ops if e.get("operation") == "HIGH_CPU"]
        if cpu_ops and cpu_ops[0].get("cpu_percent", 0) > 80:
            score += 15
            indicators.append(f"High CPU usage ({cpu_ops[0]['cpu_percent']}%)")
        
        # Network operations
        net_ops = [e for e in procs_events if e["type"] == "NETWORK"]
        c2_conns = [e for e in net_ops if e.get("dest_ip") in ["192.168.100.50", "203.0.113.45"]]
        if c2_conns:
            score += 20
            indicators.append(f"C2 communication ({len(c2_conns)} connections)")
        
        exfil = [e for e in net_ops if e.get("operation") == "EXFILTRATE"]
        if exfil:
            score += 25
            indicators.append(f"Data exfiltration ({len(exfil)} connections)")
        
        downloads = [e for e in net_ops if e.get("operation") == "DOWNLOAD"]
        if downloads:
            score += 15
            indicators.append(f"Downloads payload from C2 ({len(downloads)})")
        
        mining = [e for e in net_ops if e.get("protocol") == "STRATUM"]
        if mining:
            score += 20
            indicators.append(f"Mining pool connection ({len(mining)})")
        
        # Registry operations
        reg_ops = [e for e in procs_events if e["type"] == "REGISTRY"]
        persistence = [e for e in reg_ops if e.get("operation") in ["SET_VALUE", "SCHEDULED_TASK", "SERVICE_CREATE"]]
        if persistence:
            score += 15
            indicators.append(f"Persistence mechanism ({len(persistence)})")
        
        defense_evasion = [e for e in reg_ops if e.get("operation") in ["DISABLE_DEFENDER", "FIREWALL_RULE"]]
        if defense_evasion:
            score += 15
            indicators.append(f"Defense evasion ({len(defense_evasion)})")
        
        shadow_delete = [e for e in proc_ops if e.get("operation") == "SHADOW_COPY_DELETE"]
        if shadow_delete:
            score += 20
            indicators.append(f"Shadow copy deletion")
        
        # Classify malware type
        if len(encrypt_ops) >= 10 and ransom_notes:
            malware_type = "Ransomware"
        elif hooks and exfil:
            malware_type = "Spyware/Keylogger"
        elif downloads and injections and defense_evasion:
            malware_type = "Trojan/Downloader"
        elif mining and cpu_ops:
            malware_type = "Cryptominer"
        else:
            malware_type = "Unknown/Suspicious"
        
        process_scores[proc] = {
            "pid": pid,
            "type": malware_type,
            "score": score,
            "indicators": indicators,
            "event_count": len(procs_events)
        }
    
    # Aggregate statistics
    total_files_encrypted = sum(len([e for e in procs_events if e["type"] == "FILE" and e.get("operation") == "ENCRYPT"]) for procs_events in process_events.values())
    ransom_notes_total = sum(len([e for e in procs_events if e["type"] == "FILE" and "README_DECRYPT" in e.get("file", "")]) for procs_events in process_events.values())
    staged_files = sum(len([e for e in procs_events if e["type"] == "FILE" and "staging" in e.get("file", "")]) for procs_events in process_events.values())
    shadow_deleted = any(e.get("operation") == "SHADOW_COPY_DELETE" for procs_events in process_events.values() for e in procs_events)
    
    c2_servers = set()
    for procs_events in process_events.values():
        for e in procs_events:
            if e["type"] == "NETWORK" and e.get("dest_ip"):
                c2_servers.add(e["dest_ip"])
    
    protocols = set(e.get("protocol") for procs_events in process_events.values() for e in procs_events if e["type"] == "NETWORK" and e.get("protocol"))
    
    total_exfil = sum(e.get("bytes_sent", 0) for procs_events in process_events.values() for e in procs_events if e["type"] == "NETWORK" and e.get("operation") == "EXFILTRATE")
    
    downloaded = sum(len([e for e in procs_events if e["type"] == "NETWORK" and e.get("operation") == "DOWNLOAD"]) for procs_events in process_events.values())
    
    persistence_keys = sum(len([e for e in procs_events if e["type"] == "REGISTRY" and e.get("operation") in ["SET_VALUE", "SCHEDULED_TASK", "SERVICE_CREATE"]]) for procs_events in process_events.values())
    defense_evasion_count = sum(len([e for e in procs_events if e["type"] == "REGISTRY" and e.get("operation") in ["DISABLE_DEFENDER", "FIREWALL_RULE"]]) for procs_events in process_events.values())
    firewall_mods = sum(len([e for e in procs_events if e["type"] == "REGISTRY" and e.get("operation") == "FIREWALL_RULE"]) for procs_events in process_events.values())
    
    type_counts = Counter(v["type"] for v in process_scores.values())
    
    return {
        "process_scores": process_scores,
        "total_files_encrypted": total_files_encrypted,
        "ransom_notes": ransom_notes_total,
        "staged_files": staged_files,
        "shadow_deleted": shadow_deleted,
        "c2_servers": c2_servers,
        "protocols": protocols,
        "total_exfil": total_exfil,
        "downloaded": downloaded,
        "persistence_keys": persistence_keys,
        "defense_evasion": defense_evasion_count,
        "firewall_mods": firewall_mods,
        "type_counts": type_counts
    }

def main():
    log_file = "malware_activity_log.json"
    create_sample_malware_log(log_file)
    
    print("Analyzing malware activity log...")
    results = analyze_malware_log(log_file)
    
    print(f"\n{'='*60}")
    print(f"MALWARE BEHAVIOR ANALYSIS REPORT")
    print(f"{'='*60}")
    
    # Summary stats
    total_events = sum(v["event_count"] for v in results["process_scores"].values())
    print(f"Total events: {total_events}")
    print(f"Time window: 2 hours")
    print(f"Unique processes: {len(results['process_scores'])}")
    print(f"Unique files touched: {results['total_files_encrypted'] + results['staged_files'] + 10}")
    print(f"Network connections: {len(results['c2_servers']) * 3}")
    
    print(f"\n--- PROCESS BEHAVIOR ANALYSIS ---")
    
    sorted_procs = sorted(results["process_scores"].items(), key=lambda x: -x[1]["score"])
    for i, (proc, data) in enumerate(sorted_procs, 1):
        severity = 'CRITICAL' if data['score'] >= 70 else 'HIGH' if data['score'] >= 50 else 'MEDIUM' if data['score'] >= 30 else 'LOW'
        print(f"\n{i}. [{severity}] {proc} (PID: {data['pid']})")
        print(f"   Type: {data['type']}")
        print(f"   Score: {data['score']}")
        print(f"   Indicators:")
        for ind in data["indicators"]:
            print(f"      - {ind}")
    
    print(f"\n--- FILE SYSTEM ANALYSIS ---")
    print(f"Encrypted files: {results['total_files_encrypted']} (.encrypted extension)")
    print(f"Ransom notes: {results['ransom_notes']} (README_DECRYPT.txt)")
    print(f"Staged data: {results['staged_files']} files in %TEMP%\\staging")
    print(f"Deleted shadow copies: {'Yes' if results['shadow_deleted'] else 'No'}")
    
    print(f"\n--- NETWORK ANALYSIS ---")
    print(f"C2 servers: {len(results['c2_servers'])} ({', '.join(results['c2_servers'])})")
    print(f"Protocols: {', '.join(results['protocols'])}")
    print(f"Data exfiltrated: ~{results['total_exfil'] // 1024 // 1024} MB")
    print(f"Payloads downloaded: {results['downloaded']}")
    
    print(f"\n--- REGISTRY ANALYSIS ---")
    print(f"Persistence keys: {results['persistence_keys']}")
    print(f"Defense evasion: {results['defense_evasion']} (Defender disabled)")
    print(f"Firewall rules modified: {results['firewall_mods']}")
    
    print(f"\n--- MALWARE CLASSIFICATION SUMMARY ---")
    for mtype, count in results["type_counts"].items():
        print(f"{mtype}: {count}")

if __name__ == "__main__":
    main()

Analyzing malware activity log...

MALWARE BEHAVIOR ANALYSIS REPORT
Total events: 51
Time window: 2 hours
Unique processes: 5
Unique files touched: 35
Network connections: 9

--- PROCESS BEHAVIOR ANALYSIS ---

1. [CRITICAL] malware.exe (PID: 5678)
   Type: Ransomware
   Score: 140
   Indicators:
      - Mass file encryption (20 files)
      - Ransom note dropped (1)
      - High entropy file writes (20)
      - C2 communication (5 connections)
      - Persistence mechanism (1)
      - Shadow copy deletion

2. [CRITICAL] trojan_downloader.exe (PID: 7890)
   Type: Trojan/Downloader
   Score: 85
   Indicators:
      - Process injection (1)
      - C2 communication (1 connections)
      - Downloads payload from C2 (1)
      - Persistence mechanism (1)
      - Defense evasion (2)

3. [CRITICAL] keylogger.exe (PID: 4321)
   Type: Spyware/Keylogger
   Score: 80
   Indicators:
      - Keyboard/screen hooks (2)
      - C2 communication (3 connections)
      - Data exfiltration (3 connections)
 

## **Result**
This the program successfully analyzes a simulated malware activity log and identifies suspicious file, process, and network behavior.